# Optimization Note 4: The Primal-Dual Interior Point Method

**Goal:** Solve problems with both bounds and equality constraints in a single Newton-like framework — no nested loops.

## The Problem

$$\min_x f(x) \quad \text{s.t.} \quad c(x) = 0, \quad x \geq \ell$$

The barrier approach from Note 3 handles bounds but not equality constraints, and requires solving a full subproblem for each $\mu$. The primal-dual method combines everything into **one Newton step per iteration**.

## KKT Conditions

The first-order optimality conditions (KKT) for the barrier subproblem are:

$$\nabla f(x) + J(x)^T y - z = 0 \quad \text{(stationarity)}$$
$$c(x) = 0 \quad \text{(primal feasibility)}$$
$$Z S \mathbf{1} = \mu \mathbf{1} \quad \text{(complementarity)}$$
$$s > 0, \; z > 0 \quad \text{(positivity)}$$

where:
- $y$ = constraint multipliers (Lagrange multipliers)
- $z$ = bound multipliers
- $s = x - \ell$ = slacks to bounds
- $J(x)$ = constraint Jacobian $\nabla c(x)^T$

## The Newton Step on the KKT System

Apply Newton's method to the KKT conditions. Eliminate the complementarity equation using $z = \mu / s = \mu / (x - \ell)$, and the KKT system becomes:

$$\begin{bmatrix} H + \Sigma & J^T \\ J & 0 \end{bmatrix} \begin{bmatrix} \Delta x \\ \Delta y \end{bmatrix} = -\begin{bmatrix} \nabla f + J^T y - z \\ c(x) \end{bmatrix}$$

where $\Sigma_{ii} = z_i / s_i$ is the barrier diagonal and $H = \nabla^2 f(x) + \sum_j y_j \nabla^2 c_j(x)$ is the Hessian of the Lagrangian.

This is a **symmetric indefinite** system — exactly what LDL^T with Bunch-Kaufman (Notes 3, 6 of the linear solvers series) is designed for!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=8, suppress=True)

In [ ]:
def primal_dual_ipm(f, grad_f, hess_f, c_eq, jac_eq, hess_lag,
                    x0, lower_bounds,
                    mu_init=0.1, tol=1e-8, max_iter=100):
    """Primal-dual interior point method for:
    
        min f(x)  s.t. c(x) = 0, x >= lower
    
    Parameters:
        f, grad_f, hess_f: objective and derivatives
        c_eq: equality constraint function (returns m-vector)
        jac_eq: constraint Jacobian (returns m x n matrix)
        hess_lag: Hessian of Lagrangian (takes x, y)
        x0: initial primal point (must satisfy x0 > lower)
        lower_bounds: lower bounds (use -np.inf for unbounded)
    """
    n = len(x0)
    x = np.array(x0, dtype=float)
    lb = np.array(lower_bounds, dtype=float)
    
    # Identify bounded variables
    bounded = np.isfinite(lb)
    n_bounds = np.sum(bounded)
    
    # Initialize multipliers
    m = len(c_eq(x))
    y = np.zeros(m)               # constraint multipliers
    z = np.ones(n_bounds) * 0.1   # bound multipliers (> 0)
    mu = mu_init
    
    history = []
    
    for k in range(max_iter):
        # Current slacks
        s = x[bounded] - lb[bounded]  # > 0 by construction
        
        # Evaluate functions
        gf = grad_f(x)
        ce = c_eq(x)
        J = jac_eq(x)
        H = hess_lag(x, y)
        
        # Barrier diagonal: Sigma_ii = z_i / s_i
        sigma = np.zeros(n)
        z_full = np.zeros(n)  # bound multipliers mapped to full n-vector
        j_bound = 0
        for i in range(n):
            if bounded[i]:
                sigma[i] = z[j_bound] / s[j_bound]
                z_full[i] = z[j_bound]
                j_bound += 1
        
        # KKT residuals
        r_dual = gf + J.T @ y - z_full          # stationarity
        r_primal = ce                             # primal feasibility
        compl = np.max(np.abs(z * s - mu)) if n_bounds > 0 else 0.0
        
        # Convergence measures
        dual_inf = np.linalg.norm(r_dual, np.inf)
        primal_inf = np.linalg.norm(r_primal, np.inf)
        
        history.append({
            'x': x.copy(), 'y': y.copy(), 'z': z.copy(),
            'f': f(x), 'mu': mu,
            'dual_inf': dual_inf, 'primal_inf': primal_inf,
            'compl': compl
        })
        
        # Check convergence
        if dual_inf < tol and primal_inf < tol and compl < tol:
            break
        
        # Build KKT system
        # [H + Sigma   J^T] [dx]   [-r_dual + mu/s - z]
        # [J           0  ] [dy] = [-r_primal          ]
        
        # RHS incorporates barrier: r_dual uses z, and we add mu/s centering
        rhs_1 = -gf - J.T @ y
        j_bound = 0
        for i in range(n):
            if bounded[i]:
                rhs_1[i] += z[j_bound] - mu / s[j_bound]
                j_bound += 1
        rhs_2 = -ce
        
        # Assemble augmented system
        KKT = np.zeros((n + m, n + m))
        KKT[:n, :n] = H + np.diag(sigma)
        KKT[:n, n:] = J.T
        KKT[n:, :n] = J
        
        rhs = np.concatenate([rhs_1, rhs_2])
        
        # Solve
        sol = np.linalg.solve(KKT, rhs)
        dx = sol[:n]
        dy = sol[n:]
        
        # Compute dz from complementarity: dz = (mu - z*ds - z*s) / s
        # where ds = dx for bounded components
        dz = np.zeros(n_bounds)
        j_bound = 0
        for i in range(n):
            if bounded[i]:
                dz[j_bound] = (mu - z[j_bound] * s[j_bound] - z[j_bound] * dx[i]) / s[j_bound]
                j_bound += 1
        
        # Fraction-to-boundary rule
        tau = max(0.99, 1.0 - mu)
        alpha_primal = 1.0
        j_bound = 0
        for i in range(n):
            if bounded[i] and dx[i] < 0:
                alpha_primal = min(alpha_primal, -tau * s[j_bound] / dx[i])
            if bounded[i]:
                j_bound += 1
        
        alpha_dual = 1.0
        for j_b in range(n_bounds):
            if dz[j_b] < 0:
                alpha_dual = min(alpha_dual, -tau * z[j_b] / dz[j_b])
        
        # Update
        x += alpha_primal * dx
        y += alpha_dual * dy
        z += alpha_dual * dz
        
        # Update barrier parameter
        if n_bounds > 0:
            s_new = x[bounded] - lb[bounded]
            avg_compl = np.dot(z, s_new) / n_bounds
            mu = min(0.2 * mu, avg_compl / 10.0)
            mu = max(mu, 1e-12)
        else:
            mu = max(mu * 0.2, 1e-12)
    
    return x, y, z, history

## Example 1: Equality-Constrained QP

$$\min \; (x_1 - 2)^2 + (x_2 - 1)^2 \quad \text{s.t.} \quad x_1 + x_2 = 1, \quad x \geq 0$$

In [ ]:
f = lambda x: (x[0] - 2)**2 + (x[1] - 1)**2
grad_f = lambda x: np.array([2*(x[0] - 2), 2*(x[1] - 1)])
hess_f = lambda x: np.array([[2.0, 0.0], [0.0, 2.0]])

c = lambda x: np.array([x[0] + x[1] - 1])  # x1 + x2 = 1
J = lambda x: np.array([[1.0, 1.0]])
hess_lag = lambda x, y: hess_f(x)  # constraints are linear, no second-order term

x0 = np.array([0.5, 0.5])
lb = [0.0, 0.0]

x_opt, y_opt, z_opt, hist = primal_dual_ipm(f, grad_f, hess_f, c, J, hess_lag, x0, lb)

print(f"Solution:   x = {x_opt}")
print(f"Multiplier: y = {y_opt}  (constraint)")
print(f"Multiplier: z = {z_opt}  (bounds)")
print(f"f(x*) = {f(x_opt):.6f}")
print(f"c(x*) = {c(x_opt)}")
print(f"\nExpected: x = [0.75, 0.25], y = [-1.5], z ≈ [0, 0]")

In [ ]:
# Show convergence history
print(f"{'Iter':>4} {'f(x)':>10} {'||dual||':>10} {'||primal||':>10} {'compl':>10} {'mu':>10}")
print("-" * 58)
for i, h in enumerate(hist):
    print(f"{i:>4} {h['f']:>10.6f} {h['dual_inf']:>10.2e} {h['primal_inf']:>10.2e} {h['compl']:>10.2e} {h['mu']:>10.2e}")

## Example 2: HS035 — A Classic Test Problem

From the Hock-Schittkowski test suite (problem 35):

$$\min \; 9 - 8x_1 - 6x_2 - 4x_3 + 2x_1^2 + 2x_2^2 + x_3^2 + 2x_1 x_2 + 2x_1 x_3$$
$$\text{s.t.} \quad x_1 + x_2 + 2x_3 \leq 3, \quad x \geq 0$$

We convert the inequality to equality with a slack: $x_1 + x_2 + 2x_3 + s = 3$, $s \geq 0$.

In [ ]:
# HS035: augment with slack variable s = x[3]
def f35(x):
    return 9 - 8*x[0] - 6*x[1] - 4*x[2] + 2*x[0]**2 + 2*x[1]**2 + x[2]**2 + 2*x[0]*x[1] + 2*x[0]*x[2]

def grad_f35(x):
    return np.array([
        -8 + 4*x[0] + 2*x[1] + 2*x[2],
        -6 + 2*x[0] + 4*x[1],
        -4 + 2*x[2] + 2*x[0],
        0.0  # slack
    ])

def hess_lag35(x, y):
    H = np.zeros((4, 4))
    H[0, 0] = 4; H[0, 1] = 2; H[0, 2] = 2
    H[1, 0] = 2; H[1, 1] = 4
    H[2, 0] = 2; H[2, 2] = 2
    return H

c35 = lambda x: np.array([x[0] + x[1] + 2*x[2] + x[3] - 3])  # with slack
J35 = lambda x: np.array([[1.0, 1.0, 2.0, 1.0]])

x0 = np.array([0.5, 0.5, 0.5, 1.0])  # x + slack
lb35 = [0.0, 0.0, 0.0, 0.0]

x_opt, y_opt, z_opt, hist = primal_dual_ipm(
    f35, grad_f35, lambda x: hess_lag35(x, None)[:4,:4],
    c35, J35, hess_lag35, x0, lb35
)

print(f"Solution: x = {x_opt[:3]}")
print(f"Slack:    s = {x_opt[3]:.6f}")
print(f"f(x*) = {f35(x_opt):.6f}")
print(f"Constraint: x1+x2+2*x3 = {x_opt[0]+x_opt[1]+2*x_opt[2]:.6f} <= 3")
print(f"\nExpected: x = [4/3, 7/9, 4/9], f = 1/9 ≈ 0.1111")

## The KKT Matrix Structure

Let's visualize the structure of the KKT system. This is the matrix that gets factored at every iteration — the connection point between the optimization and linear algebra tutorial series.

$$K = \begin{bmatrix} H + \Sigma & J^T \\ J & 0 \end{bmatrix}$$

**Properties:**
- **Symmetric** ($K = K^T$)
- **Indefinite** (the $(2,2)$ block is zero/negative)
- **Expected inertia:** $(n, m, 0)$ — this is checked by the linear solver
- For large sparse problems, this is exactly what rmumps factors

In [ ]:
# Visualize the KKT matrix structure
n, m_eq = 4, 1
H = hess_lag35(x_opt, y_opt)
sigma = np.array([z_opt[i] / max(x_opt[i] - lb35[i], 1e-10) for i in range(4)])
Jmat = J35(x_opt)

KKT = np.zeros((n + m_eq, n + m_eq))
KKT[:n, :n] = H + np.diag(sigma)
KKT[:n, n:] = Jmat.T
KKT[n:, :n] = Jmat

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Structure
axes[0].spy(KKT, markersize=15)
axes[0].set_title('KKT matrix sparsity pattern')
axes[0].axhline(y=n-0.5, color='red', linestyle='--')
axes[0].axvline(x=n-0.5, color='red', linestyle='--')
axes[0].text(n/2 - 0.5, -0.8, '$H + \\Sigma$', ha='center', fontsize=12)
axes[0].text(n + 0.1, n/2 - 0.5, '$J^T$', fontsize=12)
axes[0].text(n/2 - 0.5, n + 0.3, '$J$', ha='center', fontsize=12)

# Eigenvalues
eigvals = np.linalg.eigvalsh(KKT)
colors = ['green' if e > 0 else 'red' for e in eigvals]
axes[1].bar(range(len(eigvals)), eigvals, color=colors)
axes[1].set_title('KKT eigenvalues (green=+, red=−)')
axes[1].set_xlabel('Index')
axes[1].set_ylabel('Eigenvalue')
axes[1].axhline(y=0, color='black', linewidth=0.5)

n_pos = sum(1 for e in eigvals if e > 1e-10)
n_neg = sum(1 for e in eigvals if e < -1e-10)
axes[1].text(0.5, 0.95, f'Inertia: ({n_pos}+, {n_neg}−, 0z)',
             transform=axes[1].transAxes, fontsize=12, ha='center', va='top')

plt.tight_layout()
plt.show()

print(f"Expected inertia: ({n}+, {m_eq}−, 0z) for n={n} variables, m={m_eq} constraints")

## Barrier Parameter Update

ripopt uses two strategies for decreasing $\mu$:

**Free (adaptive) mode:** $\mu_{k+1} = \text{avg\_complementarity} / \kappa$ where $\kappa = 10$

**Fixed (monotone) mode:** $\mu_{k+1} = \min(0.2 \cdot \mu_k, \, \mu_k^{1.5})$

The superlinear $\mu^{1.5}$ term accelerates convergence: once $\mu$ is small, it shrinks much faster than linear.

ripopt starts in free mode and switches to fixed when progress stalls. This adaptive strategy is one of the key differences from textbook implementations.

## What We've Learned

1. The **primal-dual IPM** solves the KKT conditions directly — one linear system per iteration
2. The KKT matrix is **symmetric indefinite** with structure $\begin{bmatrix} H+\Sigma & J^T \\ J & 0 \end{bmatrix}$
3. **Bound multipliers** $z$ and **constraint multipliers** $y$ are updated alongside $x$
4. The **fraction-to-boundary rule** keeps $x > \ell$ and $z > 0$
5. The **barrier parameter** $\mu$ decreases adaptively toward zero
6. **Inertia** of the KKT matrix must be $(n, m, 0)$ — the linear solver checks this

## What's Next

So far we've handled equality constraints and bounds. In Note 5, we extend to **nonlinear inequality constraints** $g(x) \leq 0$ — which requires slack variables, a more complex KKT system, and careful handling of constraint infeasibility.

---

*This is Optimization Note 4 in a series building from Newton's method to the interior point optimizer [ripopt](https://github.com/jkitchin/ripopt).*